# Milestone 1

In [1]:
import pandas as pd
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import string
import warnings
warnings.filterwarnings("ignore")

In [2]:
df_train = pd.read_csv("../dataset/train.csv")
df_test  = pd.read_csv("../dataset/test.csv")

In [3]:
CHOICES = ["A", "B", "C", "D", "E"]

ans_counts = df_train["answer"].value_counts()
most_freq = ans_counts.max()
least_freq = ans_counts.min()
print(f"Sum of most and least frequent option occurrences: {most_freq + least_freq}")

Sum of most and least frequent option occurrences: 814


In [4]:
def clean_and_split(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

all_unique_words = set()
for prompt in df_train["prompt"]:
    all_unique_words.update(clean_and_split(prompt))
print(f"Total unique words in cleaned prompts: {len(all_unique_words)}")

Total unique words in cleaned prompts: 859


In [5]:
if 'id' in df_train.columns:
    row_1 = df_train[(df_train['id'] == 1) | (df_train['id'] == '1')].iloc[0]
else:
    row_1 = df_train.iloc[0]

row_1_text = str(row_1["prompt"]) if pd.notna(row_1["prompt"]) else ""
row_1_words = clean_and_split(row_1_text)

filtered_row_1 = [w for w in row_1_words if w not in ENGLISH_STOP_WORDS]
print(f"Words left in Row ID 1 prompt after stop-word filtering: {len(filtered_row_1)}")

Words left in Row ID 1 prompt after stop-word filtering: 13


In [6]:
combined_text = []
for _, row in df_train.iterrows():
    combined_text.append(str(row["prompt"]))
    for c in CHOICES:
        combined_text.append(str(row[c]))

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text)
print(f"Total number of feature columns (vocab size): {len(vectorizer.get_feature_names_out())}")

Total number of feature columns (vocab size): 2762


In [7]:
p_text = str(row_1["prompt"]) if pd.notna(row_1["prompt"]) else ""
a_text = str(row_1["A"])      if pd.notna(row_1["A"])      else ""
 
prompt_vec   = vectorizer.transform([p_text])
option_a_vec = vectorizer.transform([a_text])
 
cos_sim = cosine_similarity(prompt_vec, option_a_vec)[0][0]
print(f"Cosine similarity (Row ID 1, Prompt vs Option A): {cos_sim:.4f}")

Cosine similarity (Row ID 1, Prompt vs Option A): 0.2328


In [8]:
correct_predictions = 0
for _, row in df_train.iterrows():
    p_vec = vectorizer.transform([str(row["prompt"])])
    similarities = []
    for c in CHOICES:
        opt_vec = vectorizer.transform([str(row[c])])
        sim = cosine_similarity(p_vec, opt_vec)[0][0]
        similarities.append((c, sim))
        
    similarities.sort(key=lambda x: x[1], reverse=True)
    best_guess = similarities[0][0]
    
    if best_guess == row["answer"]:
        correct_predictions += 1

accuracy = (correct_predictions / len(df_train)) * 100
print(f"Percentage where highest cosine similarity matches correct answer: {accuracy:.2f}%")

Percentage where highest cosine similarity matches correct answer: 13.70%


In [9]:
def map_at_3(ground_truth, predictions):
    score = 0.0
    for i, p in enumerate(predictions[:3]):
        if p == ground_truth:
            score = 1.0 / (i + 1)
            break
    return score

print(f"MAP@3 score if truth is C and prediction is [C, A, B]: {map_at_3('C', ['C', 'A', 'B']):.4f}")
print(f"MAP@3 score if truth is B and prediction is [D, B, E]: {map_at_3('B', ['D', 'B', 'E']):.4f}")

MAP@3 score if truth is C and prediction is [C, A, B]: 1.0000
MAP@3 score if truth is B and prediction is [D, B, E]: 0.5000


In [10]:
top_3_overall = ans_counts.head(3).index.tolist()
majority_map3_scores = [map_at_3(truth, top_3_overall) for truth in df_train["answer"]]
print(f"Overall MAP@3 score for Majority Class Baseline: {np.mean(majority_map3_scores):.4f}")

Overall MAP@3 score for Majority Class Baseline: 0.4213


In [11]:
p_vecs   = vectorizer.transform(df_train["prompt"].fillna("").astype(str))
opt_vecs = {c: vectorizer.transform(df_train[c].fillna("").astype(str)) for c in CHOICES}
 
def map_at_3(ground_truth, predictions):
    for i, p in enumerate(predictions[:3]):
        if p == ground_truth:
            return 1.0 / (i + 1)
    return 0.0
 
scores = []
for i in range(len(df_train)):
    sims = [(c, cosine_similarity(p_vecs[i], opt_vecs[c][i])[0][0]) for c in CHOICES]
    sims.sort(key=lambda x: x[1], reverse=True)
    top3 = [x[0] for x in sims[:3]]
 
    scores.append(map_at_3(df_train["answer"].iloc[i], top3))
 
print(f"TF-IDF Pipeline MAP@3: {np.mean(scores):.4f}")

TF-IDF Pipeline MAP@3: 0.3119
